In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, RobustScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
import joblib

# 1. Create realistic Telecom CDR / Network dataset
np.random.seed(42)
N = 5000
df = pd.DataFrame({
    "call_duration": np.random.exponential(scale=15.0, size=N),
    "bytes_transferred": np.random.exponential(scale=5000.0, size=N),
    "failed_attempts": np.random.poisson(lam=0.5, size=N),
    "device_brand": np.random.choice(["Apple", "Samsung", "Xiaomi", "Other", None], size=N),
    "connection_type": np.random.choice(["5G", "4G", "3G", "WiFi"], size=N),
    "is_fraud": np.random.choice([0, 1], size=N, p=[0.96, 0.04])
})

X = df.drop(columns=["is_fraud"])
y = df["is_fraud"]

# Split data with stratification
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 2. Define Feature Subsets
robust_numeric_features = ["bytes_transferred", "call_duration"] # Outlier-heavy
standard_numeric_features = ["failed_attempts"]
categorical_features = ["device_brand", "connection_type"]

# 3. Build Sub-Pipelines
robust_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", RobustScaler())
])

standard_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="mean")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="constant", fill_value="UNKNOWN")),
    ("ohe", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

# 4. Assemble Full ColumnTransformer Preprocessor
preprocessor = ColumnTransformer(
    transformers=[
        ("num_robust", robust_pipeline, robust_numeric_features),
        ("num_standard", standard_pipeline, standard_numeric_features),
        ("cat", categorical_pipeline, categorical_features)
    ],
    remainder="drop"
)

# 5. Connect Preprocessor directly to Estimator in an End-to-End Pipeline
full_model_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(class_weight="balanced", max_iter=1000, random_state=42))
])

# 6. Evaluate with Stratified K-Fold Cross-Validation
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(full_model_pipeline, X_train, y_train, cv=cv, scoring="f1")
print(f"Stratified 5-Fold F1 Score: {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})")

# 7. Fit on entire training set & evaluate on held-out test set
full_model_pipeline.fit(X_train, y_train)
y_pred = full_model_pipeline.predict(X_test)
print("\nFinal Test Classification Report:")
print(classification_report(y_test, y_pred, target_names=["Legitimate", "Fraud"]))

# 8. Export Model Artifact for FastAPI Production Serving
joblib.dump(full_model_pipeline, "telecom_fraud_pipeline.joblib")
print("Saved pipeline artifact to telecom_fraud_pipeline.joblib")


Stratified 5-Fold F1 Score: 0.0559 (+/- 0.0071)

Final Test Classification Report:
              precision    recall  f1-score   support

  Legitimate       0.98      0.55      0.70       962
       Fraud       0.05      0.66      0.10        38

    accuracy                           0.55      1000
   macro avg       0.52      0.60      0.40      1000
weighted avg       0.94      0.55      0.68      1000

Saved pipeline artifact to telecom_fraud_pipeline.joblib


In [1]:
# available dataset list in scikit-learn
from sklearn import datasets
available_datasets = [name for name in dir(datasets) if not name.startswith("_")]
available_datasets

['clear_data_home',
 'dump_svmlight_file',
 'fetch_20newsgroups',
 'fetch_20newsgroups_vectorized',
 'fetch_california_housing',
 'fetch_covtype',
 'fetch_file',
 'fetch_kddcup99',
 'fetch_lfw_pairs',
 'fetch_lfw_people',
 'fetch_olivetti_faces',
 'fetch_openml',
 'fetch_rcv1',
 'fetch_species_distributions',
 'get_data_home',
 'load_breast_cancer',
 'load_diabetes',
 'load_digits',
 'load_files',
 'load_iris',
 'load_linnerud',
 'load_sample_image',
 'load_sample_images',
 'load_svmlight_file',
 'load_svmlight_files',
 'load_wine',
 'make_biclusters',
 'make_blobs',
 'make_checkerboard',
 'make_circles',
 'make_classification',
 'make_friedman1',
 'make_friedman2',
 'make_friedman3',
 'make_gaussian_quantiles',
 'make_hastie_10_2',
 'make_low_rank_matrix',
 'make_moons',
 'make_multilabel_classification',
 'make_regression',
 'make_s_curve',
 'make_sparse_coded_signal',
 'make_sparse_spd_matrix',
 'make_sparse_uncorrelated',
 'make_spd_matrix',
 'make_swiss_roll',
 'textwrap']

In [ ]:
datasets.load_digits()


KeyError: 0